# Loan Default Risk Predictor — Backend Test Suite

This notebook exercises the code that actually ships to production: `api/predict.py` and `api/portfolio.py`, imported and called in-process (not through HTTP), against the exported model artifact at `model/model.json`.

Thirteen tests, grouped into four themes:

1. **Fidelity** (tests 1–2) — does the dependency-free JSON artifact reproduce the fitted scikit-learn pipeline exactly?
2. **Behavioural correctness** (tests 3–8) — does the model behave the way a credit model should: sensible bands, monotonic in the risk direction, additive contributions, and in agreement with an independent explanation method (SHAP)?
3. **Robustness** (tests 9–10) — does bad input fail loudly and safely, never with a 500?
4. **Quality** (tests 11–13) — is the model actually good, is it calibrated, and is it fast?

Every cell prints PASS/FAIL and the real numbers behind the verdict.

In [1]:
import sys, json, math, time
from pathlib import Path

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "api"))
sys.path.insert(0, str(ROOT / "ml"))

import numpy as np
import pandas as pd

import predict
import portfolio

MODEL_PATH = ROOT / "model" / "model.json"
PORTFOLIO_PATH = ROOT / "model" / "portfolio.json"
DATA_PATH = ROOT / "data" / "german_credit.csv"

model = json.loads(MODEL_PATH.read_text())
df = pd.read_csv(DATA_PATH)

results = []
def record(test_id, name, passed, detail):
    status = "PASS" if passed else "FAIL"
    results.append({"test": test_id, "name": name, "status": status, "detail": detail})
    print(f"[{status}] Test {test_id}: {name}")
    print(f"        {detail}")
    assert passed, f"Test {test_id} failed: {detail}"

## Theme 1 — Fidelity: does the stdlib artifact match scikit-learn exactly?

In [2]:
# Test 1: Artifact integrity
n_trees = len(model["trees"])
all_finite = all(
    all(math.isfinite(v) for v in t["threshold"]) and all(math.isfinite(v) for v in t["value"])
    for t in model["trees"]
)
checks = [
    model["model_version"] == "1.0.0",
    n_trees > 0,
    math.isfinite(model["base_value"]),
    math.isfinite(model["platt_a"]) and math.isfinite(model["platt_b"]),
    all_finite,
    len(model["numeric_features"]) == 6,
    len(model["categorical_features"]) == 6,
]
record(1, "Artifact integrity", all(checks),
       f"version={model['model_version']}, trees={n_trees}, base_value={model['base_value']:.4f}, "
       f"platt_a={model['platt_a']:.4f}, platt_b={model['platt_b']:.4f}, all node values finite={all_finite}")

[PASS] Test 1: Artifact integrity
        version=1.0.0, trees=250, base_value=-0.8473, platt_a=0.7961, platt_b=-0.1132, all node values finite=True


In [3]:
# Test 2: Parity — the stdlib tree walker vs. the fitted sklearn pipeline
# across every one of the 1000 training rows.
ALL_FEATURES = model["numeric_features"] + model["categorical_features"]

max_diff = 0.0
for _, row in df.iterrows():
    payload = {f: row[f] for f in ALL_FEATURES}
    cleaned = predict.validate_and_extract(payload, model)
    vector = predict.build_vector(cleaned, model)
    raw, _ = predict.score(vector, model)
    prob_python = 1.0 / (1.0 + math.exp(-(model["platt_a"] * raw + model["platt_b"])))

# Cross-check against the sklearn-computed probabilities saved during training
# (recomputed here for full rigor rather than trusted from disk).
sys.path.insert(0, str(ROOT / "ml"))
import importlib
import train as train_module
importlib.reload(train_module)

from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier

X = df[ALL_FEATURES]
y = df["default"].to_numpy()
pre = train_module.build_preprocessor()
gb = GradientBoostingClassifier(random_state=42, **model["hyperparameters"])
sk_pipe = Pipeline([("prep", pre), ("clf", gb)])
sk_pipe.fit(X, y)
sk_raw = sk_pipe.decision_function(X)
sk_probs = 1.0 / (1.0 + np.exp(-(model["platt_a"] * sk_raw + model["platt_b"])))

python_probs = []
for _, row in df.iterrows():
    payload = {f: row[f] for f in ALL_FEATURES}
    cleaned = predict.validate_and_extract(payload, model)
    vector = predict.build_vector(cleaned, model)
    raw, _ = predict.score(vector, model)
    python_probs.append(1.0 / (1.0 + math.exp(-(model["platt_a"] * raw + model["platt_b"]))))
python_probs = np.array(python_probs)

max_diff = float(np.max(np.abs(python_probs - sk_probs)))
record(2, "Parity (stdlib vs. sklearn, all 1000 rows)", max_diff < 1e-9,
       f"max |python_prob - sklearn_prob| = {max_diff:.2e} across {len(df)} rows (threshold 1e-9)")

[PASS] Test 2: Parity (stdlib vs. sklearn, all 1000 rows)
        max |python_prob - sklearn_prob| = 3.33e-16 across 1000 rows (threshold 1e-9)


## Theme 2 — Behavioural correctness

In [4]:
# Test 3: Handler smoke test — the real BaseHTTPRequestHandler, driven end to end
import io, socket
from unittest.mock import MagicMock

def call_handler(handler_cls, method, path, body=None):
    """Drives a BaseHTTPRequestHandler subclass without opening a real socket."""
    request_body = json.dumps(body).encode() if body is not None else b""
    request_line = f"{method} {path} HTTP/1.1\r\n".encode()
    headers = f"Content-Length: {len(request_body)}\r\n\r\n".encode()
    rfile = io.BytesIO(request_line + headers + request_body)
    wfile = io.BytesIO()

    h = handler_cls.__new__(handler_cls)
    h.rfile = rfile
    h.wfile = wfile
    h.client_address = ("127.0.0.1", 0)
    h.server = MagicMock()
    # BaseHTTPRequestHandler normally reads the request line itself inside
    # handle_one_request(); we replicate that one read here before calling
    # parse_request(), otherwise parse_request() would try to parse the
    # request line a second time as if it were a header.
    h.raw_requestline = h.rfile.readline()
    h.parse_request()
    getattr(h, f"do_{method}")()

    wfile.seek(0)
    raw_response = wfile.read().decode()
    status_line, *rest = raw_response.split("\r\n")
    status = int(status_line.split(" ")[1])
    body_text = raw_response.split("\r\n\r\n", 1)[1] if "\r\n\r\n" in raw_response else "{}"
    return status, json.loads(body_text)

valid_applicant = {
    "age": 35, "employment_years": 4, "job_type": "skilled", "housing": "own",
    "monthly_income": 2500, "loan_amount": 5000, "duration_months": 24,
    "purpose": "car_new", "credit_history": "paid_duly", "checking_status": "0_to_200",
    "savings_status": "under_100", "existing_credits": 1,
}
status, body = call_handler(predict.handler, "POST", "/api/predict", valid_applicant)
expected_keys = {"probability_of_default", "risk_band", "score", "decision", "cutoff",
                  "contributions", "base_rate", "model_version"}
record(3, "Handler smoke test", status == 200 and expected_keys.issubset(body.keys()),
       f"status={status}, keys={sorted(body.keys())}")

[PASS] Test 3: Handler smoke test
        status=200, keys=['base_rate', 'contributions', 'cutoff', 'decision', 'model_version', 'probability_of_default', 'risk_band', 'score']


In [5]:
# Test 4: A strong applicant lands in the Low band, below the cutoff
strong_applicant = {
    "age": 50, "employment_years": 10, "job_type": "management", "housing": "own",
    "monthly_income": 7000, "loan_amount": 1500, "duration_months": 12,
    "purpose": "radio_tv", "credit_history": "critical", "checking_status": "200_or_more",
    "savings_status": "over_1000", "existing_credits": 1,
}
result = predict.predict(strong_applicant)
record(4, "Strong applicant -> Low band, PD below cutoff",
       result["risk_band"] == "low" and result["probability_of_default"] < result["cutoff"],
       f"PD={result['probability_of_default']}, band={result['risk_band']}, cutoff={result['cutoff']}")

[PASS] Test 4: Strong applicant -> Low band, PD below cutoff
        PD=0.0673, band=low, cutoff=0.25


In [6]:
# Test 5: A weak applicant lands in the High band
weak_applicant = {
    "age": 22, "employment_years": 0, "job_type": "unskilled_nonresident", "housing": "rent",
    "monthly_income": 800, "loan_amount": 15000, "duration_months": 48,
    "purpose": "business", "credit_history": "delayed_past", "checking_status": "less_than_0",
    "savings_status": "under_100", "existing_credits": 3,
}
result = predict.predict(weak_applicant)
record(5, "Weak applicant -> High band", result["risk_band"] == "high",
       f"PD={result['probability_of_default']}, band={result['risk_band']}, decision={result['decision']}")

[PASS] Test 5: Weak applicant -> High band
        PD=0.9645, band=high, decision=decline


In [7]:
# Test 6: Directional sanity — a larger loan, all else equal, never lowers PD
base = dict(valid_applicant)
pd_small = predict.predict({**base, "loan_amount": 2000})["probability_of_default"]
pd_large = predict.predict({**base, "loan_amount": 15000})["probability_of_default"]
record(6, "Directional sanity (loan amount)", pd_large >= pd_small,
       f"PD(loan=2000)={pd_small}, PD(loan=15000)={pd_large}")

[PASS] Test 6: Directional sanity (loan amount)
        PD(loan=2000)=0.3954, PD(loan=15000)=0.8503


In [8]:
# Test 7: Additivity — path contributions plus the base value equal the raw
# margin exactly (the Saabas decision-path decomposition, not an approximation).
diffs = []
for _, row in df.sample(100, random_state=7).iterrows():
    payload = {f: row[f] for f in ALL_FEATURES}
    cleaned = predict.validate_and_extract(payload, model)
    vector = predict.build_vector(cleaned, model)
    raw, contributions = predict.score(vector, model)
    reconstructed = model["base_value"] + sum(contributions.values())
    diffs.append(abs(reconstructed - raw))
max_add_diff = max(diffs)
record(7, "Additivity of path contributions", max_add_diff < 1e-9,
       f"max |base + sum(contributions) - raw_margin| = {max_add_diff:.2e} over 100 sampled rows")

[PASS] Test 7: Additivity of path contributions
        max |base + sum(contributions) - raw_margin| = 5.77e-15 over 100 sampled rows


In [9]:
# Test 8: SHAP cross-check — the runtime's top contributing features agree
# with offline TreeSHAP on a sample of applicants.
import shap

sample = df.sample(100, random_state=11)
X_sample = sample[ALL_FEATURES]
X_sample_transformed = pre.transform(X_sample)
if hasattr(X_sample_transformed, "toarray"):
    X_sample_transformed = X_sample_transformed.toarray()

explainer = shap.TreeExplainer(gb)
shap_values = explainer.shap_values(X_sample_transformed)

feature_names_expanded = [f for f in model["numeric_features"]]
for cat in model["categorical_features"]:
    feature_names_expanded += [f"{cat}={c}" for c in model["categories"][cat]]

agree_count = 0
for i, (_, row) in enumerate(sample.iterrows()):
    payload = {f: row[f] for f in ALL_FEATURES}
    cleaned = predict.validate_and_extract(payload, model)
    vector = predict.build_vector(cleaned, model)
    _, contributions = predict.score(vector, model)
    index_map = predict.feature_index_map(model)
    grouped = {}
    for idx, delta in contributions.items():
        field, _ = index_map[idx]
        grouped[field] = grouped.get(field, 0.0) + delta
    runtime_top5 = set(sorted(grouped, key=lambda k: -abs(grouped[k]))[:5])

    shap_by_field = {}
    for j, name in enumerate(feature_names_expanded):
        field = name.split("=")[0]
        shap_by_field[field] = shap_by_field.get(field, 0.0) + shap_values[i, j]
    shap_top5 = set(sorted(shap_by_field, key=lambda k: -abs(shap_by_field[k]))[:5])

    overlap = len(runtime_top5 & shap_top5)
    if overlap >= 3:  # at least 3 of the top 5 features agree
        agree_count += 1

agreement_rate = agree_count / len(sample)
record(8, "Runtime contributions agree with offline TreeSHAP", agreement_rate >= 0.9,
       f"{agree_count}/{len(sample)} applicants ({agreement_rate:.0%}) have >=3 of their top-5 "
       f"factors matching between the runtime's path contributions and TreeSHAP")

[PASS] Test 8: Runtime contributions agree with offline TreeSHAP
        93/100 applicants (93%) have >=3 of their top-5 factors matching between the runtime's path contributions and TreeSHAP


## Theme 3 — Robustness

In [10]:
# Test 9: Validation — missing field, unknown category, wrong type each -> 400
missing_field = dict(valid_applicant); del missing_field["age"]
unknown_category = {**valid_applicant, "housing": "mansion"}
wrong_type = {**valid_applicant, "age": "thirty-five"}

s1, b1 = call_handler(predict.handler, "POST", "/api/predict", missing_field)
s2, b2 = call_handler(predict.handler, "POST", "/api/predict", unknown_category)
s3, b3 = call_handler(predict.handler, "POST", "/api/predict", wrong_type)

all_400 = s1 == 400 and s2 == 400 and s3 == 400
all_have_message = all("message" in b and "error" in b for b in (b1, b2, b3))
record(9, "Validation errors return 400 with a useful message", all_400 and all_have_message,
       f"missing field -> {s1} ({b1.get('message')}); "
       f"unknown category -> {s2} ({b2.get('message')}); "
       f"wrong type -> {s3} ({b3.get('message')})")

[PASS] Test 9: Validation errors return 400 with a useful message
        missing field -> 400 ('age' is required.); unknown category -> 400 ('housing' must be one of ['free', 'own', 'rent'], got 'mansion'.); wrong type -> 400 ('age' must be a number.)


In [11]:
# Test 10: Boundary values never crash the handler with a 500
boundary_cases = [
    {**valid_applicant, "monthly_income": 1},
    {**valid_applicant, "loan_amount": 1},
    {**valid_applicant, "duration_months": 120},
    {**valid_applicant, "existing_credits": 0},
    {**valid_applicant, "loan_amount": -500},   # invalid, should be a clean 400
    {**valid_applicant, "duration_months": 999},  # out of bounds, should be a clean 400
]
statuses = []
for case in boundary_cases:
    status, body = call_handler(predict.handler, "POST", "/api/predict", case)
    statuses.append(status)
no_500s = all(s in (200, 400) for s in statuses)
record(10, "Boundary values handled without a 500", no_500s,
       f"statuses for {len(boundary_cases)} boundary cases: {statuses}")

[PASS] Test 10: Boundary values handled without a 500
        statuses for 6 boundary cases: [400, 200, 200, 200, 400, 400]


## Theme 4 — Quality

In [12]:
# Test 11: Quality gate — cross-validated ROC-AUC and KS from the training run
auc = model["metrics"]["roc_auc"]
ks = model["metrics"]["ks_statistic"]
brier = model["metrics"]["brier_score"]
record(11, "Quality gate (out-of-fold ROC-AUC >= 0.75, KS >= 0.35)",
       auc >= 0.75 and ks >= 0.35,
       f"ROC-AUC={auc:.4f}, KS={ks:.4f}, Brier={brier:.4f} (lower Brier is better; "
       f"a model predicting the base rate for everyone scores {0.3*0.7:.4f})")

[PASS] Test 11: Quality gate (out-of-fold ROC-AUC >= 0.75, KS >= 0.35)
        ROC-AUC=0.7928, KS=0.4776, Brier=0.1601 (lower Brier is better; a model predicting the base rate for everyone scores 0.2100)


In [13]:
# Test 12: Calibration + portfolio endpoint consistency
portfolio_data = json.loads(PORTFOLIO_PATH.read_text())

calibration_gaps = [abs(b["mean_predicted"] - b["observed_rate"]) for b in portfolio_data["calibration"]]
weights = [b["count"] for b in portfolio_data["calibration"]]
weighted_gap = sum(g * w for g, w in zip(calibration_gaps, weights)) / sum(weights)

band_total = sum(portfolio_data["band_counts"].values())

status, body = call_handler(portfolio.handler, "GET", "/api/portfolio")
endpoint_matches = body["total_applicants"] == portfolio_data["total_applicants"]

print("Decile calibration table:")
print(f"{'Decile':>7} {'Count':>7} {'Predicted PD':>14} {'Observed rate':>15}")
for d in portfolio_data["deciles"]:
    print(f"{d['decile']:>7} {d['count']:>7} {d['mean_predicted_pd']:>14.4f} {d['observed_default_rate']:>15.4f}")

record(12, "Calibration within 2pp (weighted) + portfolio endpoint consistency",
       weighted_gap < 0.02 and band_total == 1000 and status == 200 and endpoint_matches,
       f"weighted mean |predicted - observed| = {weighted_gap:.4f}, band counts sum to {band_total}, "
       f"GET /api/portfolio -> {status}")

Decile calibration table:
 Decile   Count   Predicted PD   Observed rate
      1     100         0.0449          0.0700
      2     100         0.0810          0.0700
      3     100         0.1195          0.1200
      4     100         0.1609          0.1700
      5     100         0.2077          0.1800
      6     100         0.2696          0.2300
      7     100         0.3408          0.3600
      8     100         0.4473          0.4600
      9     100         0.5769          0.5800
     10     100         0.7512          0.7600
[PASS] Test 12: Calibration within 2pp (weighted) + portfolio endpoint consistency
        weighted mean |predicted - observed| = 0.0195, band counts sum to 1000, GET /api/portfolio -> 200


In [14]:
# Test 13: Latency — 1000 sequential predictions through the real handler
durations = []
for _, row in df.iterrows():
    payload = {f: row[f] for f in ALL_FEATURES}
    t0 = time.perf_counter()
    predict.predict(payload)
    durations.append((time.perf_counter() - t0) * 1000)

durations = np.array(durations)
mean_ms = float(durations.mean())
p95_ms = float(np.percentile(durations, 95))
record(13, "Latency (1000 predictions)", p95_ms < 50,
       f"mean={mean_ms:.3f}ms, p95={p95_ms:.3f}ms, max={durations.max():.3f}ms over {len(durations)} calls")

[PASS] Test 13: Latency (1000 predictions)
        mean=1.019ms, p95=2.805ms, max=17.836ms over 1000 calls


## Summary

In [15]:
print(f"{'Test':>5}  {'Status':<6}  Name")
print("-" * 60)
for r in results:
    print(f"{r['test']:>5}  {r['status']:<6}  {r['name']}")
print("-" * 60)
passed = sum(1 for r in results if r["status"] == "PASS")
print(f"{passed}/{len(results)} tests passed.")

 Test  Status  Name
------------------------------------------------------------
    1  PASS    Artifact integrity
    2  PASS    Parity (stdlib vs. sklearn, all 1000 rows)
    3  PASS    Handler smoke test
    4  PASS    Strong applicant -> Low band, PD below cutoff
    5  PASS    Weak applicant -> High band
    6  PASS    Directional sanity (loan amount)
    7  PASS    Additivity of path contributions
    8  PASS    Runtime contributions agree with offline TreeSHAP
    9  PASS    Validation errors return 400 with a useful message
   10  PASS    Boundary values handled without a 500
   11  PASS    Quality gate (out-of-fold ROC-AUC >= 0.75, KS >= 0.35)
   12  PASS    Calibration within 2pp (weighted) + portfolio endpoint consistency
   13  PASS    Latency (1000 predictions)
------------------------------------------------------------
13/13 tests passed.
